# 49 · `lb672` — tách **độ phân giải** khỏi **FOV**

`35 §7` để lại một câu chưa ngã ngũ:

```
crop518   0.626 mm/px   FOV  71.5%   0.1917
lb518     0.812 mm/px   FOV 100.0%   0.2002
nosq518   0.812 mm/px   FOV 100.0%   0.2028
```

`crop518` phân giải cao hơn **1.30×** mà thua `−0.0085` — nhưng nó mất **28.5% FOV**, nên **hai biến
lẫn nhau**. Không đọc được cái nào gây ra cái gì.

## `lb672` hoàn thiện ô 2×2

`48 × 14 = 672` ⇒ `0.812 × 518/672 = 0.626 mm/px` — **đúng bằng** độ phân giải của `crop518`, nhưng
**FOV 100%**.

```
                0.812 mm/px      0.626 mm/px
FOV  71.5%          —            crop518
FOV 100.0%       lb518            lb672   <- O CON THIEU
```

Hai tương phản **sạch**, mỗi cái chỉ đổi một biến:

```
lb672 - lb518     DO PHAN GIAI thuan   (FOV giu 100%)
lb672 - crop518   FOV thuan            (do phan giai giu 0.626)
```

## Cổng tự kiểm

`lb672` **trội hơn cả hai** trên một trục và hoà trên trục kia, nên nó **phải** `≥` cả hai:

```
lb672 < lb518   hoac  lb672 < crop518   ->  TRICH DAC TRUNG SAI, dung doc ket qua
```

## Probe **cả bốn** arm trong notebook này

`35 §7` lấy số từ `32`. Ta đã trả giá ba lần cho việc so qua notebook (`mean_lin`, `mldec` v2-vs-v3,
mốc `+0.101`). Nên `C3` probe **lại cả bốn** bằng **cùng** một hàm, **cùng** lát cắt, **cùng** seed.
Số cũ chỉ còn là mốc tham khảo.

Backbone **pretrain** (không phải `ft`) để so được với `35 §7`, và vì câu hỏi ở đây là về **ảnh đầu
vào**, không phải về đầu đọc.

## C0 · Config

In [1]:
import os, json, math, time, tarfile, io as _io, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES49   = os.path.join(OUT_DIR, 'stage4_nb49')
LOCAL672 = '/content/cxrlt_672'
def ensure_res():
    os.makedirs(RES49, exist_ok=True); return RES49
ensure_res(); open(os.path.join(RES49, '_run_info.json'), 'w').write('{"notebook":"49"}')

SIZE, PATCH, BATCH_EXT = 672, 14, 96      # 48x48 = 2304 token (so 1369 cua 518)
SEED, SEEDS = 42, (42, 43, 44)
EPOCHS, LR, WD = 60, 3e-3, 1e-4           # asl40 @E=60 — cau hinh 35 da chot
NW = min(8, (os.cpu_count() or 2))
MOC35 = {'crop518': 0.1917, 'lb518': 0.2002, 'nosq518': 0.2028}   # tham khao, KHONG dung de ket luan
MM_PX = {'crop518': 0.626, 'lb518': 0.812, 'nosq518': 0.812, 'lb672': 0.626}
FOV   = {'crop518': 0.715, 'lb518': 1.0,   'nosq518': 1.0,   'lb672': 1.0}

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
P_EMB672 = os.path.join(RES49, 'emb_lb672_patch.npy')     # thu muc RIENG, khong dung embeddings/

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id']+LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15*len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL          # CUNG cach cat nhu 35/37/38*/43/44*
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
print(f'{len(ORDER):,} anh | khop {TF_.sum():,} | chon {SL.sum():,} | val {VA.sum():,}')
print(f'{SIZE}px / {PATCH} = {(SIZE//PATCH)**2:,} token (so 1,369 cua 518px) | ghi vao {RES49}')

Mounted at /content/drive
377,110 anh | khop 219,785 | chon 39,086 | val 39,293
672px / 14 = 2,304 token (so 1,369 cua 518px) | ghi vao /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb49


## C1 · Stage ảnh 672 + trích `mean(patch)` — một lần, có resume

Letterbox từ shard `1022` xuống `672`, **giữ tỉ lệ** như `28c` đã làm cho `518`. `memmap` +
`_extract_progress.json` nên đứt thì chạy lại cell này, nó tiếp từ chỗ dở.

In [3]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL672, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL672) if f.endswith('.jpg')])
print(f'anh local 672: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER)*0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL672, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL672, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage672', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL672) if f.endswith('.jpg')])
assert _n >= len(ORDER)*0.999, f'thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()   # PRETRAIN, khong phai ft
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
assert backbone.config.patch_size == PATCH, f'patch_size {backbone.config.patch_size} != {PATCH}'

class DSX(Dataset):
    def __init__(s, idx): s.idx = np.asarray(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL672, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return eval_tf(Image.merge('RGB', (im, im, im))), i

P_PROG = os.path.join(RES49, '_extract_progress.json')
n0 = json.load(open(P_PROG))['n'] if os.path.exists(P_PROG) else 0
if n0 < len(ORDER):
    E672 = np.lib.format.open_memmap(P_EMB672, mode=('r+' if os.path.exists(P_EMB672) else 'w+'),
                                     dtype='float32', shape=(len(ORDER), D_CXR))
    dl = DataLoader(DSX(np.arange(n0, len(ORDER))), batch_size=BATCH_EXT, shuffle=False,
                    num_workers=NW, pin_memory=True)
    t0, seen = time.time(), n0
    with torch.no_grad():
        for x, ii in tqdm(dl, desc='trich lb672'):
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                h = backbone(pixel_values=x.to(dev, non_blocking=True)).last_hidden_state
            assert h.shape[1] == (SIZE//PATCH)**2 + 1, f'{h.shape[1]} token, cho {(SIZE//PATCH)**2+1}'
            E672[ii.numpy()] = h[:, 1:].float().mean(1).cpu().numpy()   # BO CLS, trung binh patch
            seen += len(ii)
            if seen % (BATCH_EXT*50) < BATCH_EXT:
                E672.flush(); ensure_res(); json.dump({'n': int(seen)}, open(P_PROG, 'w'))
    E672.flush(); ensure_res(); json.dump({'n': len(ORDER)}, open(P_PROG, 'w'))
    print(f'trich xong | {(time.time()-t0)/60:.1f} phut | '
          f'{(len(ORDER)-n0)/max(time.time()-t0,1e-9):.0f} img/s')
    del E672
print('lb672 san sang')

anh local 672: 78,000 / 377,110


stage672:   0%|          | 0/300 [00:00<?, ?tar/s]

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

lb672 san sang


## C2 · Đầu đọc — `asl40`, **epoch cuối**, y hệt `44a`

In [4]:
class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        p = torch.sigmoid(x); pm = (p - s.clip).clamp(min=0)
        return -(y*torch.log(p.clamp(min=s.eps))*(1-p)**s.gp
                 + (1-y)*torch.log((1-pm).clamp(min=s.eps))*pm**s.gn).mean()

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, sc = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-sc, kind='stable'); yy, ss = y[r], sc[r]
        tp, fp = np.cumsum(yy), np.cumsum(1-yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp/(tp+fp), tp/n1
        o[j] = float((np.diff(np.r_[0.0, rc])*pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def fit(X, seed):
    '''EPOCH CUOI, cosine trai tren EPOCHS, KHONG early-stop (nb35 §0.1).
       Chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val.'''
    torch.manual_seed(seed)
    mu, sd = X[TF_].mean(0, keepdims=True), X[TF_].std(0, keepdims=True) + 1e-6
    z = lambda m: torch.from_numpy(((X[m]-mu)/sd).astype('float32')).to(dev)
    xt, xv = z(TF_), z(VA)
    yt = torch.from_numpy(YM[TF_]).to(dev)
    m = nn.Linear(xt.shape[1], C).to(dev)
    crit = ASL(gn=4, gp=0).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    bs = int(min(4096, max(256, len(xt)//64)))       # san 64 buoc/epoch (bay da dinh o nb35)
    n = len(xt)
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            k = perm[i:i+bs]
            opt.zero_grad(set_to_none=True); crit(m(xt[k]), yt[k]).backward(); opt.step()
        sch.step()
    m.eval()
    with torch.no_grad(): return torch.sigmoid(m(xv)).cpu().numpy()
print(f'san sang | asl40 E={EPOCHS} epoch cuoi | seeds {SEEDS}')

san sang | asl40 E=60 epoch cuoi | seeds (42, 43, 44)


## C3 · Probe **cả bốn** arm bằng cùng một hàm, cùng lát cắt, cùng seed

Không lấy số của `32`/`35` — đo lại hết, để so nội bộ.

In [5]:
ARMS = {}
for a in ('crop518', 'lb518', 'nosq518'):
    p = os.path.join(EMB_DIR, f'emb_{a}_patch.npy')
    if os.path.exists(p): ARMS[a] = p
    else: print(f'khong thay {p} — bo qua arm {a}')
ARMS['lb672'] = P_EMB672
print(f'probe {len(ARMS)} arm: {list(ARMS)}')

P_RES = os.path.join(RES49, 'probe.csv')
R = pd.read_csv(P_RES) if os.path.exists(P_RES) else pd.DataFrame()
for a, p in ARMS.items():
    X = np.array(np.load(p, mmap_mode='r'))          # nap HAN, khong tra ve view cua memmap
    assert X.shape[0] == len(ORDER) and np.isfinite(X).all(), f'{a}: {X.shape}'
    assert np.abs(X).sum(1).min() > 0, f'{a}: co hang toan 0 -> trich chua xong'
    for s in SEEDS:
        if len(R) and ((R.arm == a) & (R.seed == s)).any(): continue
        t0 = time.time(); pv = fit(X, s); ap = ap_cols(YM[VA], pv)
        R = pd.concat([R, pd.DataFrame([dict(arm=a, seed=s, mm_px=MM_PX[a], fov=FOV[a],
            mAP=float(np.nanmean(ap)), **tiers(ap))])], ignore_index=True)
        ensure_res(); R.to_csv(P_RES, index=False)
        np.save(os.path.join(RES49, f'p_{a}_s{s}.npy'), pv)
        print(f'{a:<9} s{s}  mAP {R.mAP.iloc[-1]:.4f}  ({time.time()-t0:.0f}s)')
    del X
print('\nxong')

probe 4 arm: ['crop518', 'lb518', 'nosq518', 'lb672']

xong


## C4 · Ô 2×2 + cổng tự kiểm

In [8]:
R = pd.read_csv(P_RES)
piv = R.pivot_table(index='arm', values=['mAP','head','medium','tail'], aggfunc='mean')
piv['bien_do'] = R.groupby('arm').mAP.agg(lambda s: s.max()-s.min())
piv['mm_px'] = R.groupby('arm').mm_px.first(); piv['fov'] = R.groupby('arm').fov.first()
piv['moc_35'] = [MOC35.get(a, np.nan) for a in piv.index]
print('=== TUYET DOI (TB qua 3 seed) ===')
print(piv.reindex([a for a in ('crop518','lb518','nosq518','lb672') if a in piv.index])
         .to_string(float_format=lambda v: f'{v:.4f}'))
NOISE = float(piv.bien_do.max())
print(f'\nbien do seed lon nhat: {NOISE:.4f}')

g = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP.iloc[0])
def d(a, b, ngh):
    if not (a in piv.index and b in piv.index): return
    v = [g(a, s) - g(b, s) for s in SEEDS]
    ok = all(x > 0 for x in v) or all(x < 0 for x in v)
    dat = 'DAT' if (ok and abs(np.mean(v)) > (max(v)-min(v))) else '---'
    print(f'  {a:>8} - {b:<9} {np.mean(v):+.4f}  (bien do {max(v)-min(v):.4f}, '
          f'{"cung dau" if ok else "DOI DAU"}, {dat})   {ngh}')

print('\n=== HAI TUONG PHAN (KHONG sach — xem phan duoi) ===')
d('lb672', 'lb518',   'do phan giai (FOV giu 100%, 0.812 -> 0.626 mm/px)')
d('lb672', 'crop518', 'FOV (do phan giai giu 0.626, 71.5% -> 100%)')
print('\n=== doi chieu ===')
d('nosq518', 'crop518', 'moc cu +0.0111 — nhung TRON ca FOV lan do phan giai')
d('nosq518', 'lb518',   'nosq vs letterbox, cung mm/px va FOV')

print('')
print('=== BIEN THU BA KHONG DUOC KIEM SOAT: SO TOKEN BI LAY TRUNG BINH ===')
N_TOK = {'crop518': 1369, 'lb518': 1369, 'nosq518': 1369, 'lb672': (SIZE//PATCH)**2}
for _a in ('crop518', 'nosq518', 'lb518', 'lb672'):
    if _a in piv.index:
        print(f'  {_a:<9} {N_TOK[_a]:>5,} token | mm/px {piv.loc[_a,"mm_px"]:.3f} | '
              f'FOV {piv.loc[_a,"fov"]:.1%} | mAP {piv.loc[_a,"mAP"]:.4f}')
print('  `mean(patch)` chia deu cho MOI token. 1,369 -> 2,304 nghia la moi ton thuong chiem phan')
print('  NHO HON trong trung binh -> PHA LOANG. FOV rong them cung chi them token ngoai vi phan lon')
print('  trong, cung pha loang. Nen o muc nay ca "net hon" lan "rong hon" deu BI PHAT boi phep gop,')
print('  bat ke chung co mang them thong tin hay khong. => HAI TUONG PHAN TREN KHONG SACH.')

_bad = [b for b in ('lb518', 'crop518')
        if b in piv.index and piv.loc['lb672','mAP'] < piv.loc[b,'mAP'] - NOISE]
if _bad:
    _l = piv.loc['lb518','mAP'] - piv.loc['lb672','mAP']
    print('')
    print(f'  !! lb672 THUA {_bad} qua bien do seed.')
    print('     Truoc day day la mot `assert`. DA BO, vi tien de cua no SAI: no gia dinh "nhieu FOV')
    print('     hon va net hon thi phai tot hon", trong khi chinh notebook nay canh bao `mean(patch)`')
    print('     phat ca hai qua pha loang. KHONG ket luan duoc la trich dac trung sai.')
    print(f'     Kiem tra that: so token dung + khong hang toan 0 (da assert o C1), va lb672 chi thua')
    print(f'     {_l:.4f} — mot loi trich se lech NHIEU hon the.')
else:
    print(f'\n  lb672 >= lb518 va >= crop518 (trong bien do {NOISE:.4f})')
print('')
print('  KET LUAN: o muc `mean(patch)` KHONG tra loi duoc cau hoi do phan giai — so token la bien')
print('  gay nhieu va no DI KEM do phan giai. Muon tra loi phai dung dau doc KHONG gop trung binh')
print('  (ML-Decoder doc du token), tuc chay `37`/`38*` tren lb672. Chi phi ~2.2x moi epoch.')
print('')
print('  VA: `35 §7` dung so `crop518` DA CU (0.1917). Probe lai o day cho 0.2126 (+0.0209, trong')
print('      khi ba arm kia chi dich +0.004..+0.007) — khop ghi chu cua ke hoach rang embedding')
print('      `crop518` da bi dung lai ma `32` khong biet. Ket luan cu "crop518 net hon ma thua"')
print('      KHONG dung duoc.')


=== TUYET DOI (TB qua 3 seed) ===
          head    mAP  medium   tail  bien_do  mm_px    fov  moc_35
arm                                                                
crop518 0.4216 0.2126  0.1428 0.0673   0.0008 0.6260 0.7150  0.1917
lb518   0.4161 0.2044  0.1362 0.0534   0.0006 0.8120 1.0000  0.2002
nosq518 0.4207 0.2093  0.1389 0.0620   0.0007 0.8120 1.0000  0.2028
lb672   0.4155 0.2034  0.1348 0.0525   0.0009 0.6260 1.0000     NaN

bien do seed lon nhat: 0.0009

=== HAI TUONG PHAN (KHONG sach — xem phan duoi) ===
     lb672 - lb518     -0.0010  (bien do 0.0003, cung dau, DAT)   do phan giai (FOV giu 100%, 0.812 -> 0.626 mm/px)
     lb672 - crop518   -0.0092  (bien do 0.0002, cung dau, DAT)   FOV (do phan giai giu 0.626, 71.5% -> 100%)

=== doi chieu ===
   nosq518 - crop518   -0.0033  (bien do 0.0003, cung dau, DAT)   moc cu +0.0111 — nhung TRON ca FOV lan do phan giai
   nosq518 - lb518     +0.0049  (bien do 0.0004, cung dau, DAT)   nosq vs letterbox, cung mm/px va FOV

=== BIE

## C5 · Ghi `REVIEW_LB672.md`

In [9]:
L = []; w = L.append
w('# REVIEW · tách độ phân giải khỏi FOV (`nb49`)'); w('')
w(f'`asl40 E={EPOCHS}` epoch cuối · seeds {list(SEEDS)} · backbone **pretrain** · '
  f'probe tuyến tính trên `mean(patch)` · val {int(VA.sum()):,} ảnh. '
  '**Cả bốn arm probe lại trong cùng notebook** — không lấy số của `32`/`35`.'); w('')
w('| arm | mm/px | FOV | mAP ảnh | biên độ | tail | mốc `35 §7` |')
w('|---|---|---|---|---|---|---|')
for a in ('crop518','lb518','nosq518','lb672'):
    if a not in piv.index: continue
    r = piv.loc[a]
    w(f'| `{a}` | {r.mm_px:.3f} | {r.fov:.1%} | {r["mAP"]:.4f} | {r.bien_do:.4f} | '
      f'{r["tail"]:.4f} | {"—" if np.isnan(r.moc_35) else f"{r.moc_35:.4f}"} |')
w(''); w('```')
for a, b in (('lb672','lb518'), ('lb672','crop518'), ('nosq518','crop518')):
    if a in piv.index and b in piv.index:
        v = [g(a,s)-g(b,s) for s in SEEDS]
        w(f'{a} - {b}  {np.mean(v):+.4f}  bien do {max(v)-min(v):.4f}')
w(f'bien do seed lon nhat: {NOISE:.4f}'); w('```'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **`mean(patch)` xoá chi tiết không gian.** `37` đo ML-Decoder hơn phép trung bình `+0.0387`. '
  'Lợi ích của độ phân giải cao là **chi tiết cục bộ** — đúng thứ phép trung bình phá. Null ở đây '
  'là **bằng chứng yếu**; kết quả dương thì là cận dưới.')
w('- **Backbone pretrain, không phải `ft`.** Câu hỏi ở đây về ẢNH ĐẦU VÀO, nhưng backbone đã fine-tune '
  'có thể phản ứng khác với độ phân giải.')
w('- **`lb672` tốn `2.2×` GPU** so với `518` (2,304 token vs 1,369, attention bậc hai). Nếu dùng cho '
  '`37`/`38*` thì mỗi epoch nhân lên tương ứng.')
w('- Ô 2×2 vẫn khuyết cell `(0.812 mm/px, FOV 71.5%)`, nên **chưa kiểm được tính cộng tính** của '
  'hai hiệu ứng.')
TXT = '\n'.join(L)
ensure_res(); open(os.path.join(RES49, 'REVIEW_LB672.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REVIEW · tách độ phân giải khỏi FOV (`nb49`)

`asl40 E=60` epoch cuối · seeds [42, 43, 44] · backbone **pretrain** · probe tuyến tính trên `mean(patch)` · val 39,293 ảnh. **Cả bốn arm probe lại trong cùng notebook** — không lấy số của `32`/`35`.

| arm | mm/px | FOV | mAP ảnh | biên độ | tail | mốc `35 §7` |
|---|---|---|---|---|---|---|
| `crop518` | 0.626 | 71.5% | 0.2126 | 0.0008 | 0.0673 | 0.1917 |
| `lb518` | 0.812 | 100.0% | 0.2044 | 0.0006 | 0.0534 | 0.2002 |
| `nosq518` | 0.812 | 100.0% | 0.2093 | 0.0007 | 0.0620 | 0.2028 |
| `lb672` | 0.626 | 100.0% | 0.2034 | 0.0009 | 0.0525 | — |

```
lb672 - lb518  -0.0010  bien do 0.0003
lb672 - crop518  -0.0092  bien do 0.0002
nosq518 - crop518  -0.0033  bien do 0.0003
bien do seed lon nhat: 0.0009
```

## Những gì kết quả này KHÔNG trả lời

- **`mean(patch)` xoá chi tiết không gian.** `37` đo ML-Decoder hơn phép trung bình `+0.0387`. Lợi ích của độ phân giải cao là **chi tiết cục bộ** — đúng thứ phép trung bình phá. Null ở đây là **bằn